<a href="https://colab.research.google.com/github/Lee-hosuk/game-agent/blob/main/llm-wordle/llm_wordle.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🟩🟨⬜ LLM 워들 에이전트

5글자 영어 단어를 **6번 안에** 맞히는 게임입니다. 추측할 때마다 글자별로 힌트가 나옵니다.
- 🟩 글자도 자리도 맞음 · 🟨 있는 글자지만 자리가 틀림 · ⬜ 없는 글자

## 틱택토 LLM과 뭐가 다를까?
| | 틱택토 | 워들 |
|---|---|---|
| 판단 근거 | **지금 판**만 보면 됨 | **지난 힌트 전부**를 기억하고 종합해야 함 |
| 필요한 능력 | 한 수 고르기 | 여러 조건을 동시에 만족하는 단어 추론 |

이번 노트북에서는 **LLM**과 **정보 이론 봇**을 같은 문제로 비교하고,
**힌트를 코드로 정리해서 알려주면** LLM이 얼마나 좋아지는지 실험합니다.
코랩에 내장된 `google.colab.ai`를 사용하므로 **API 키가 필요 없습니다.**

## 1. 단어 목록
자주 쓰이는 5글자 영어 단어 547개입니다. 정답은 이 목록에서 나옵니다.
(인터넷에서 받지 않고 노트북에 직접 넣어서, 실행할 때마다 같은 목록을 씁니다.)

In [ ]:
WORDS = """
    about their there which would other after first think could these where
    right being going still never those world great while every state three
    since under thing house place again found might money night until doing
    group women start today point music power water small white later order
    party thank using black whole maybe story least early local video young
    court given level often death south known large wrong along class close
    cause happy human woman leave north watch light short taken third among
    check heart child major media phone quite final front ready bring heard
    march study clear month board field fight force issue price space total
    above share sense break event sorry guess learn alone movie press worth
    sound value round stand stuff drive green match model trust range trade
    chief lower style blood stage title enjoy cover legal seven staff super
    union began built crazy daily paper voice whose earth below offer sleep
    table truth build piece visit wrote river shall speak write album eight
    funny peace spent store track ahead allow brown radio cross speed extra
    quick agree clean photo scene spend coach heavy train claim hotel judge
    brain floor image meant reach civil dance stock worst beach older color
    dream grand sweet touch doubt drink shown basic carry crime fully plant
    worse award block avoid catch coast trial truly queen broke glass prior
    royal serve begin twice worry brand count fresh mouth owner scale score
    smart throw click faith metal agent apply enter prime waste weird adult
    grade stone apart aware fifth proud birth labor lucky prove tough alive
    apple dress horse magic stick frank guide raise treat youth exist guard
    shoot solid sport taste hello plane shape upper cheap joint steel crowd
    enemy limit ocean proof chair honor email stuck sugar drama entry grant
    grown lying route smoke squad teach abuse angry laugh lunch storm actor
    chain equal quiet split urban fault phase smile truck blame dying false
    novel shirt cream giant mayor minor theme usual draft drunk spoke upset
    cycle aside empty prize rural admit favor frame guest teeth bunch fruit
    index shift stood tight crown grace chose crash depth dirty error noise
    panel plate shame cable print sight broad drawn motor solar blind brief
    chest debut paint pilot steam exact quote swear trend bread inner reply
    tower wheel pride rough smell topic sharp trick audio chart mount occur
    argue awful bound cloud loose moral shock thick threw angel flash gross
    refer trail uncle badly chase ideal clock label pitch pizza plain anger
    comic ghost newly skill solve trash boost crack knife saint steal buddy
    delay elite forth layer marry nurse rugby sheet bench idiot medal shell
    silly angle forum grass hence rapid sixth valid vital creek fraud honey
    ratio scary spell sword begun daddy drove glory juice logic strip dozen
    fancy input knock ought flood rally swing alert arena brave fleet robin
    shake shore blast pound punch fewer grave mouse piano craft devil sauce
    slide tiger alien beast candy fifty relax alarm globe spare opera quest
    anime lover panic scope alpha brick mercy outer screw venue habit naval
    noble robot toxic wider wound asset baker brush essay maker snake ultra
    usage canal climb fever fluid nasty twist unity blown fence stole dealt
    flesh forty laser loyal patch radar villa
""".split()
print(len(WORDS), "개 /", WORDS[:10])

## 2. 힌트 계산하기 ⭐
같은 글자가 여러 번 나올 때가 까다롭습니다. 예를 들어 정답이 `ABIDE`인데 `SPEED`를 넣으면,
`E`가 두 번 있지만 정답에는 `E`가 하나뿐이라 **첫 번째 E만 🟨**, 두 번째 E는 ⬜입니다.

그래서 ① 먼저 🟩(자리까지 맞은 글자)를 표시하고, ② 남은 글자 개수를 세서 그만큼만 🟨를 줍니다.

In [ ]:
import random
import re
import time
import math
from collections import Counter

EMOJI = {"G": "🟩", "Y": "🟨", "B": "⬜"}

def feedback(guess, answer):
    """G(초록) / Y(노랑) / B(회색) 5글자 문자열을 돌려줌"""
    result = ["B"] * 5
    left = Counter()                        # 🟩가 아닌 정답 글자들의 개수
    for i in range(5):
        if guess[i] == answer[i]:
            result[i] = "G"
        else:
            left[answer[i]] += 1
    for i in range(5):
        if result[i] == "B" and left[guess[i]] > 0:
            result[i] = "Y"
            left[guess[i]] -= 1
    return "".join(result)

def show(guess, fb):
    return f"{guess.upper()}  " + "".join(EMOJI[c] for c in fb)

for g, a in [("speed", "abide"), ("crane", "crane"), ("alert", "later")]:
    print(f"정답 {a.upper()} / 추측 {show(g, feedback(g, a))}")

## 3. 한 판 진행하는 함수
플레이어는 `(지금까지의 기록, 남은 후보 단어들) → 추측 단어`를 돌려주는 함수입니다.
**남은 후보**는 지금까지의 힌트와 모두 맞아떨어지는 단어들이에요. (봇은 이걸 쓰고, LLM은 안 씁니다)

In [ ]:
def play(player, answer, max_turns=6, verbose=False):
    history, candidates = [], WORDS[:]
    for turn in range(1, max_turns + 1):
        guess = player(history, candidates).lower()
        fb = feedback(guess, answer)
        history.append((guess, fb))
        if verbose:
            print(f"{turn}. {show(guess, fb)}   (남은 후보 {len([w for w in candidates if feedback(guess, w) == fb])}개)")
        if fb == "GGGGG":
            return turn, history
        candidates = [w for w in candidates if feedback(guess, w) == fb]
    return None, history          # 못 맞힘

## 4. 비교용 봇 두 가지

**🎲 랜덤 후보 봇**: 지금까지 힌트에 맞는 단어 중 **아무거나** 고릅니다.

**🧠 정보 이론 봇**: 정답 후보를 **가장 많이 줄여줄 것 같은** 단어를 고릅니다.
어떤 단어를 넣었을 때 나올 수 있는 힌트 모양(🟩🟨⬜ 조합)별로 후보가 몇 개씩 남는지 세고,
**엔트로피**(평균적으로 얻는 정보량, 단위: 비트)가 가장 큰 단어를 고릅니다.
- 힌트가 골고루 갈라질수록 → 어떤 힌트가 나오든 후보가 많이 줄어듦 → 엔트로피 큼
- 1비트 = 후보가 절반으로 줄어듦

In [ ]:
def random_bot(history, candidates):
    return random.choice(candidates)

def entropy(guess, candidates):
    counts = Counter(feedback(guess, a) for a in candidates)    # 힌트 모양별 후보 개수
    n = len(candidates)
    return -sum(c / n * math.log2(c / n) for c in counts.values())

def entropy_bot(history, candidates):
    if not history:
        return FIRST_GUESS                  # 첫 단어는 미리 계산해 둠
    if len(candidates) <= 2:
        return candidates[0]
    # 정보량이 같으면 정답일 수도 있는 단어를 우선
    return max(WORDS, key=lambda g: (entropy(g, candidates), g in candidates))

t0 = time.time()
ranked = sorted(WORDS, key=lambda g: entropy(g, WORDS), reverse=True)
FIRST_GUESS = ranked[0]
print(f"계산 {time.time()-t0:.1f}초")
print("첫 단어로 좋은 단어:", [(w, round(entropy(w, WORDS), 2)) for w in ranked[:5]])
print("첫 단어로 나쁜 단어:", [(w, round(entropy(w, WORDS), 2)) for w in ranked[-3:]])

In [ ]:
random.seed(3)
answer = random.choice(WORDS)
print(f"정답: {answer.upper()}\n\n🎲 랜덤 후보 봇")
play(random_bot, answer, verbose=True)
print("\n🧠 정보 이론 봇")
play(entropy_bot, answer, verbose=True);

## 5. 봇 성능 측정
100문제를 풀게 해서 **맞힌 비율**과 **평균 몇 번 만에** 맞혔는지 봅니다. (30초 안팎)

In [ ]:
def evaluate(player, answers, label):
    turns = [play(player, a)[0] for a in answers]
    solved = [t for t in turns if t]
    avg = sum(solved) / len(solved) if solved else float("nan")
    print(f"{label:<14} | 맞힘 {len(solved)}/{len(answers)} | 평균 {avg:.2f}번 | 분포 {dict(sorted(Counter(t or 'X' for t in turns).items(), key=str))}")
    return turns

random.seed(0)
bot_answers = random.sample(WORDS, 100)
results = {
    "random bot": evaluate(random_bot, bot_answers, "🎲 랜덤 후보"),
    "entropy bot": evaluate(entropy_bot, bot_answers, "🧠 정보 이론"),
}

## 6. LLM 연결하기
사용 가능한 모델은 계정마다 다를 수 있으니 먼저 목록을 확인합니다.

In [ ]:
from google.colab import ai

print(ai.list_models())

In [ ]:
MODEL = None   # None이면 기본 모델. 위 목록에서 골라 넣어도 됨

def ask_llm(prompt):
    return ai.generate_text(prompt, model_name=MODEL) if MODEL else ai.generate_text(prompt)

print(ask_llm("Wordle 규칙을 한 문장으로 설명해줘."))

## 7. LLM 플레이어 만들기 ⭐

두 가지 프롬프트를 준비합니다.
- **기본 프롬프트**: 규칙과 지금까지의 추측·힌트만 보여줌. 힌트를 종합하는 건 LLM 몫
- **정리 프롬프트**: 힌트를 **코드로 미리 정리**해서 함께 알려줌
  (예: "3번째 자리는 A 확정", "L은 있지만 2번째 자리는 아님", "R, T, S는 없음")

LLM은 글을 글자 단위가 아니라 **토큰**(단어 조각) 단위로 읽기 때문에, "3번째 글자가 뭐지?" 같은 걸 생각보다 자주 틀립니다.
정리 프롬프트는 이 약점을 코드가 메워주는 방식이에요.

LLM이 지키지 못한 경우도 셉니다.
- **형식 오류**: 5글자 영어 단어가 아님 → 이유를 알려주고 다시 물어봄 (3번 실패하면 남은 후보 중 랜덤)
- **힌트 무시**: 이미 없다고 나온 글자를 쓰는 등, 지금까지 힌트와 맞지 않는 단어를 냄

In [ ]:
RULES = """너는 Wordle을 풀고 있어. 정답은 흔한 5글자 영어 단어야.
추측할 때마다 글자별로 힌트가 나와:
🟩 = 글자와 자리 모두 맞음, 🟨 = 정답에 있는 글자지만 자리가 틀림, ⬜ = 정답에 없는 글자
"""

def history_text(history):
    if not history:
        return "(아직 추측 없음)"
    lines = []
    for i, (g, fb) in enumerate(history, 1):
        detail = ", ".join(f"{c.upper()}{EMOJI[f]}" for c, f in zip(g, fb))
        lines.append(f"{i}. {g.upper()}  {''.join(EMOJI[f] for f in fb)}   ({detail})")
    return "\n".join(lines)

def summarize(history):
    """지금까지의 힌트를 사람이 읽기 쉬운 조건으로 정리"""
    green = ["_"] * 5
    present, not_here, absent = set(), {}, set()
    for g, fb in history:
        for i, (c, f) in enumerate(zip(g, fb)):
            if f == "G":
                green[i] = c.upper()
            elif f == "Y":
                present.add(c)
                not_here.setdefault(c, set()).add(i + 1)
    present -= {c.lower() for c in green if c != "_"}       # 이미 자리가 확정된 글자는 빼기
    for g, fb in history:
        for c, f in zip(g, fb):
            if f == "B" and c not in present and c.upper() not in green:
                absent.add(c)
    lines = [f"- 확정된 자리: {' '.join(green)}"]
    if present:
        lines.append("- 있지만 자리가 틀린 글자: " +
                     ", ".join(f"{c.upper()} ({'/'.join(map(str, sorted(not_here[c])))}번째 자리는 아님)" for c in sorted(present)))
    if absent:
        lines.append(f"- 정답에 없는 글자 (쓰지 마): {' '.join(sorted(c.upper() for c in absent))}")
    return "\n".join(lines)

def make_prompt(history, helped):
    p = RULES + f"\n지금까지의 추측:\n{history_text(history)}\n"
    if helped and history:
        p += f"\n힌트를 정리하면:\n{summarize(history)}\n위 조건을 모두 만족하는 단어를 골라.\n"
    p += f"\n남은 기회: {6 - len(history)}번\n짧게 생각을 말하고, 마지막 줄에 반드시 'GUESS: 단어' 형식으로 답해."
    return p

llm_stats = Counter()

def make_llm_player(helped, verbose=False, max_tries=3):
    def llm_player(history, candidates):
        prompt = make_prompt(history, helped)
        for _ in range(max_tries):
            llm_stats["calls"] += 1
            try:
                answer = ask_llm(prompt)
            except Exception as e:
                print("LLM 오류:", e); time.sleep(5); continue
            m = re.findall(r"GUESS\s*[:：]\s*\**\s*([A-Za-z]+)", answer)
            guess = m[-1].lower() if m else ""
            if len(guess) == 5 and guess.isalpha():
                if any(feedback(g, guess) != fb for g, fb in history):
                    llm_stats["ignored_hint"] += 1        # 지금까지 힌트와 안 맞는 단어
                if verbose:
                    print(f"🤖 {answer.strip()}\n")
                return guess
            llm_stats["format_error"] += 1
            prompt += f"\n\n'{guess or answer[-30:]}'은(는) 5글자 영어 단어가 아니야. 마지막 줄에 'GUESS: 다섯글자단어' 형식으로 다시 답해."
        llm_stats["fallback"] += 1
        return random.choice(candidates)
    return llm_player

random.seed(1)
demo = [("alert", feedback("alert", "chart")), ("chain", feedback("chain", "chart"))]
print(make_prompt(demo, helped=True))

## 8. LLM 한 판 지켜보기
정리 프롬프트를 쓰는 LLM이 한 문제를 풉니다. LLM의 생각도 같이 출력돼요.

In [ ]:
random.seed(7)
answer = random.choice(WORDS)
llm_stats.clear()
turns, hist = play(make_llm_player(helped=True, verbose=True), answer, verbose=True)
print(f"정답: {answer.upper()} → {'%d번 만에 맞힘' % turns if turns else '못 맞힘'}")
print(dict(llm_stats))

## 9. 실험: 기본 프롬프트 vs 정리 프롬프트 🧪
같은 문제들을 두 프롬프트로 풀게 합니다.

⏱️ 한 문제에 LLM을 4~6번 부르므로, 10문제 × 2가지면 **5~10분** 걸릴 수 있어요.
무료 사용량 제한에 걸리면 `N_LLM`을 줄여주세요.

In [ ]:
N_LLM = 10
random.seed(42)
llm_answers = random.sample(WORDS, N_LLM)

for helped, label in [(False, "llm basic"), (True, "llm helped")]:
    llm_stats.clear()
    t0 = time.time()
    results[label] = evaluate(make_llm_player(helped), llm_answers, "🤖 " + ("정리 프롬프트" if helped else "기본 프롬프트"))
    print(f"   LLM 호출 {llm_stats['calls']}번 | 형식 오류 {llm_stats['format_error']}번 | 힌트 무시 {llm_stats['ignored_hint']}번 | {time.time()-t0:.0f}초")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = list(results)
bins = [1, 2, 3, 4, 5, 6, None]
names = ["1", "2", "3", "4", "5", "6", "fail"]
x = np.arange(len(bins))
width = 0.8 / len(labels)

plt.figure(figsize=(9, 4))
for k, lab in enumerate(labels):
    turns = results[lab]
    pct = [100 * sum(1 for t in turns if t == b) / len(turns) for b in bins]
    solved = [t for t in turns if t]
    avg = sum(solved) / len(solved) if solved else float("nan")
    plt.bar(x + (k - (len(labels) - 1) / 2) * width, pct, width, label=f"{lab} (avg {avg:.2f})")
plt.xticks(x, names)
plt.xlabel("guesses needed")
plt.ylabel("% of games")
plt.title("Wordle: bots vs LLM")
plt.legend()
plt.savefig("wordle_results.png", dpi=100, bbox_inches="tight")
plt.show()

**볼 포인트**
- 정보 이론 봇은 후보 목록을 **직접 다 계산**하니까 빠르고 정확합니다. LLM은 정답 목록을 모르고, 머릿속으로 조건을 따져야 합니다.
- **힌트 무시** 횟수를 보세요. 기본 프롬프트에서는 이미 없다고 나온 글자를 다시 쓰는 실수가 자주 나옵니다.
- 정리 프롬프트에서 실수가 줄었다면, **LLM이 약한 부분(글자 단위 추론)을 코드가 대신 해준 효과**입니다. 실제 LLM 에이전트도 이렇게 도구와 함께 씁니다.

## 10. 직접 해보기 🎮
단어를 입력하고 **제출**을 누르세요. 막히면 정보 이론 봇에게 추천을 받을 수 있어요.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

TILE = {"G": "#6aaa64", "Y": "#c9b458", "B": "#787c7e"}

def play_wordle():
    state = {"answer": random.choice(WORDS), "history": []}
    box = widgets.Text(placeholder="5글자 영어 단어", layout=widgets.Layout(width="160px"))
    submit = widgets.Button(description="제출", button_style="primary", layout=widgets.Layout(width="70px"))
    hint = widgets.Button(description="🧠 봇 추천", layout=widgets.Layout(width="100px"))
    new = widgets.Button(description="🔄 새 게임", layout=widgets.Layout(width="100px"))
    board = widgets.HTML()
    status = widgets.HTML()

    def candidates():
        return [w for w in WORDS if all(feedback(g, w) == fb for g, fb in state["history"])]

    def render(msg=""):
        rows = ""
        for g, fb in state["history"]:
            rows += "<div style='display:flex;gap:5px;margin-bottom:5px'>" + "".join(
                f"<div style='width:44px;height:44px;background:{TILE[f]};color:white;display:flex;align-items:center;"
                f"justify-content:center;font:bold 22px sans-serif'>{c.upper()}</div>" for c, f in zip(g, fb)) + "</div>"
        for _ in range(6 - len(state["history"])):
            rows += "<div style='display:flex;gap:5px;margin-bottom:5px'>" + "".join(
                "<div style='width:44px;height:44px;border:2px solid #d3d6da;box-sizing:border-box'></div>" for _ in range(5)) + "</div>"
        board.value = rows
        status.value = f"<div style='font-size:15px'>{msg}</div>"

    def done():
        h = state["history"]
        return bool(h) and (h[-1][1] == "GGGGG" or len(h) == 6)

    def on_submit(_):
        if done():
            return
        g = box.value.strip().lower()
        if len(g) != 5 or not g.isalpha():
            render("5글자 영어 단어를 입력하세요."); return
        state["history"].append((g, feedback(g, state["answer"])))
        box.value = ""
        if state["history"][-1][1] == "GGGGG":
            render(f"🎉 {len(state['history'])}번 만에 맞혔어요!")
        elif len(state["history"]) == 6:
            render(f"😢 아쉬워요. 정답은 <b>{state['answer'].upper()}</b>")
        else:
            render(f"남은 후보 {len(candidates())}개")

    def on_hint(_):
        if done():
            return
        c = candidates()
        render(f"🧠 봇 추천: <b>{entropy_bot(state['history'], c).upper()}</b> (남은 후보 {len(c)}개)")

    def on_new(_):
        state["answer"], state["history"] = random.choice(WORDS), []
        render("새 게임! 첫 단어를 입력하세요.")

    submit.on_click(on_submit); hint.on_click(on_hint); new.on_click(on_new)
    box.on_submit(on_submit) if hasattr(box, "on_submit") else None
    display(widgets.VBox([widgets.HBox([box, submit, hint, new]), status, board]))
    render("첫 단어를 입력하세요. (정답은 위의 547개 단어 중 하나예요)")

play_wordle()

## 🔥 도전 과제
1. **프롬프트 개선**: 기본 프롬프트에 "단어를 정하기 전에 각 자리 글자를 하나씩 확인해" 같은 지시를 넣으면 힌트 무시가 줄어들까요?
2. **후보 알려주기**: 남은 후보 단어 목록을 LLM에게 직접 보여주면 어떻게 될까요? (이건 "도구를 쓰는 LLM 에이전트"의 시작입니다)
3. **LLM 첫 단어**: LLM은 첫 단어로 무엇을 고르나요? 정보 이론 봇이 고른 단어와 비교해보세요.
4. **모델 비교**: `MODEL`을 바꿔서 어떤 모델이 워들을 가장 잘 푸는지 비교해보세요.
5. **어려운 모드**: 🟩🟨로 나온 글자를 다음 추측에 반드시 써야 하는 규칙을 넣으면 봇의 평균이 어떻게 달라질까요?